# CalMS21: train a pose model from tracks, then track with TREx

A complete loop on one downloadable dataset -- **published keypoints -> pseudo-annotations
-> a pose model -> TREx tracking with visual identity -> an annotated video**. Nothing is
labelled by hand.

The dataset ships three resident-intruder recordings, 80 k-means-selected frames from each,
and the CalMS21 arrays the keypoints come from. Everything below is rebuilt from those
three things.

```
tracks_raw/*.npy  --convert-->  tracks/          7 MARS keypoints per mouse, per frame
                                    |
media/frames/  --join by frame index--
                                    v
                               pose_dataset/     a YOLO pose training set
                                    v
                               models/           a trained pose model
                                    v
                     TREx, using that model as its detector
                                    v
                               tracks/           a second variant, same clips
```

The last step is what makes this a loop rather than a line: the tracker ends up running a
model this dataset produced.

## Why pseudo-annotations are worth having

CalMS21 ships MARS-derived keypoints for every frame. Treating them as ground truth for a
*new* pose model is bootstrapping, not annotation -- the labels are as good as MARS was, no
better. That is the honest framing, and it is also the point: the training path runs end to
end without anyone labelling a frame.

## What you need

mosaic, plus [TREx](https://trex.run) in its own environment for section 6 onward. Training
in section 5 additionally needs an Ultralytics environment; without one the notebook says
so and carries on, and section 6 then tracks without a detector.

**No model weights ship with this dataset.** The trainer, Ultralytics, is AGPL-3.0. Rather
than work through what distributing a model produced by it requires of the distributor,
this example carries none and shows you how to make your own -- or point `PRETRAINED_MODEL`
at a `best.pt` you already have.

## The data

Three resident-intruder assays from CalMS21 task 1: a black resident and a white intruder in
a home cage, filmed from above at 30 fps, 13k-21k frames each. Seven MARS keypoints per
mouse (nose, ears, neck, hips, tail base).

**Citation.** Sun JJ, Karigo T, Chakraborty D, Mohanty SP, Wild B, Sun Q, Chen C, Anderson
DJ, Perona P, Yue Y, Kennedy A (2021) *The Multi-Agent Behavior Dataset: Mouse Dyadic Social
Interactions.* NeurIPS Datasets and Benchmarks.
[arXiv:2104.02710](https://arxiv.org/abs/2104.02710). The keypoints are MARS
(Segalin et al. 2021, *eLife* 10:e63720).

## 0 -- Configuration

In [ ]:
from pathlib import Path
from typing import Optional

# ---- where the data comes from ---------------------------------------------
#
#   "download" -- fetch the dataset from Hugging Face (~1.9 GB). This is what
#                 most people want.
#   "local"    -- a mosaic dataset you already have; set LOCAL_DATASET.
SOURCE = "download"

HF_REPO = "EcodylicScience/mosaic-example-calms21-pose"
HF_ASSET = "calms21-pose.tar.gz"
# Where the archive is unpacked. None -> ./calms21-pose-example
DOWNLOAD_DIR: Optional[Path] = None
# SOURCE="local": the dataset directory (the one holding dataset.yaml).
LOCAL_DATASET: Optional[Path] = None

# Everything this notebook writes lands under the dataset and is regenerable.
# True drops it all and starts over.
RESET_DERIVED = False

# ---- CalMS21 is MARS 7-keypoint --------------------------------------------
KEYPOINT_NAMES = ("nose", "left_ear", "right_ear", "neck",
                  "left_hip", "right_hip", "tail")
SKELETON = ((0, 3), (1, 0), (2, 0), (3, 4), (3, 5), (4, 6), (5, 6))
# Left<->right swap under a horizontal flip. Without this in data.yaml,
# Ultralytics silently sets fliplr=flipud=0 and trains with no flip at all.
FLIP_IDX = [0, 2, 1, 3, 5, 4, 6]

# The frames in the dataset were sampled with exactly these settings. Asking for
# them again is a cache hit rather than a re-decode -- see section 3.
N_FRAMES     = 80
FRAME_METHOD = "kmeans"
N_TRAIN_VIDS = 2        # of 3; the rest becomes valid. There is no test split.

# ---- section 5: training ----------------------------------------------------
# Training needs an Ultralytics environment and, realistically, a GPU. Set
# PRETRAINED_MODEL to a best.pt you already have to skip it -- either one gives
# section 6 a detector.
TRAIN = True
PRETRAINED_MODEL: Optional[Path] = None

BASE_MODEL = "yolo11n-pose.pt"
EPOCHS     = 200    # with patience=50; the run behind this example stopped at 183
IMGSZ      = 640
BATCH      = 16
DEVICE     = "0"    # CUDA index on a GPU box, "mps" on Apple Silicon, else "cpu"
AUGMENT    = "medium"   # none | light | medium | heavy

# ---- section 6: tracking ----------------------------------------------------
# mosaic drives TREx as a separate program and locates it on a five-step ladder:
# this argument, MOSAIC_TREX_CONDA_ENV, MOSAIC_TREX_BIN, then $PATH. Naming the
# conda environment is the reliable spelling -- TREx relaunches itself and
# resolves its embedded Python from CONDA_PREFIX, which a bare $PATH binary does
# not set. None leaves the environment variables in charge.
TREX_CONDA_ENV: Optional[str] = None

TREX_ENTRIES = 1            # how many clips to track. None = all three.
# TREx converts the whole video to a `.pv` before it tracks anything, and that
# file is close to raw -- about 320 KB a frame here, so a full 21,000-frame clip
# costs ~6.8 GB on disk. A range keeps a first run to minutes and hundreds of
# megabytes. None = the whole recording.
TREX_CONVERSION_RANGE = (0, 1799)   # 60 s at 30 fps

CLIP_SECONDS = 30       # length of the annotated video in section 8

In [ ]:
import json
import shutil
import subprocess
import tarfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mosaic.core.dataset import open_dataset
from mosaic.core.helpers import make_entry_key
from mosaic.core.pipeline.tracks_index import read_tracks_index
from mosaic.core.pipeline.ops import run_op
from mosaic.core.pipeline.inventory import inventory

from mosaic.tracking import extract_frames, list_frame_runs, get_frame_manifests
import mosaic.tracking.ops  # noqa: F401 -- registers train-pose / trex; run_op raises without it

# The canonical annotation layer: one in-memory representation of "labelled
# frames", with the YOLO writer as one emitter off it.
from mosaic.core.annotations.model import (
    AnnotationFrame, AnnotationObject, AnnotationSet, Keypoint, KeypointSchema,
)
from mosaic.core.annotations.bbox import BboxPolicy
from mosaic.tracking.pose_training.converters.emit import (
    usable_frames, write_split_tree, yolo_pose_line,
)
from mosaic.tracking.pose_training import make_data_yaml

print("imports OK")

## 1 -- Get the dataset

A mosaic dataset is a directory with a `dataset.yaml` in it, so opening one is
`open_dataset(path)` and nothing else. Reach for that rather than `Dataset(path)`: the bare
constructor **reads nothing** -- it takes a manifest *path*, so it is also how you point at
a dataset that does not exist yet, and every accessor on an unloaded one then fails against
a manifest file that is perfectly correct.

Everything this notebook computes is written back into that directory, addressed by a hash
of what produced it. All of it is regenerable, so deleting any of it costs time and never
data.

In [ ]:
# ---- resolve DATASET --------------------------------------------------------
if SOURCE == "download":
    try:
        from huggingface_hub import hf_hub_download
    except ImportError as exc:  # pragma: no cover - environment guidance
        raise ImportError(
            "SOURCE='download' needs huggingface_hub:\n"
            "    pip install 'huggingface_hub>=1.2.0'\n"
            "The version floor matters: older clients retry a rate-limit "
            "response with a 25-second backoff against a 5-minute window, so "
            "they fail rather than wait. Set SOURCE='local' to point at a copy "
            "you already have."
        ) from exc

    root = Path(DOWNLOAD_DIR) if DOWNLOAD_DIR else Path.cwd() / "calms21-pose-example"
    root.mkdir(parents=True, exist_ok=True)
    DATASET = root / "calms21-pose"
    if DATASET.exists():
        print(f"have      {DATASET}")
    else:
        print(f"fetching  {HF_ASSET} (~1.9 GB) ...")
        archive = hf_hub_download(HF_REPO, HF_ASSET, repo_type="dataset")
        with tarfile.open(archive) as tar:
            tar.extractall(root)

elif SOURCE == "local":
    if LOCAL_DATASET is None:
        raise ValueError("SOURCE='local' needs LOCAL_DATASET set to a mosaic dataset "
                         "directory (the one holding dataset.yaml).")
    DATASET = Path(LOCAL_DATASET)

else:
    raise ValueError(f"SOURCE must be 'download' or 'local', not {SOURCE!r}")

assert (DATASET / "dataset.yaml").exists(), f"no dataset.yaml under {DATASET}"

if RESET_DERIVED:
    for name in ("tracks", "models", "features", "_tracking", "pose_dataset"):
        shutil.rmtree(DATASET / name, ignore_errors=True)
    print("dropped   the derived roots")

ds = open_dataset(DATASET)
print(f"dataset   {DATASET}")
print(f"          {ds.name}, manifest v{ds.manifest.manifest_version}")
for kind in ("media", "tracks", "labels"):
    for src in ds.scan_sources(kind):
        print(f"  source  {kind:7s} {src.id:16s} path={src.path!r}")

### What is in it

Three roots, and the sections below rebuild everything else from them:

- `tracks_raw/` -- the CalMS21 task-1 arrays for these three recordings (18 MB)
- `media_raw/` -- the three clips, AV1, analysis-clean
- `media/frames/` -- 80 k-means-selected frames per clip, with their manifests

Two things are **deliberately absent**. `tracks/` is regenerable from `tracks_raw/`, and an
absent tracks index reads as an empty frame carrying the full schema rather than raising.
`pose_dataset/` -- the YOLO training tree -- is a rearrangement of the frames and the
tables, the same pixels under different filenames, so shipping it would enlarge the download
to carry nothing new. Rebuilding both is sections 2 and 4, and is the part of the recipe you
would want to change anyway.

`inventory(ds)` is the general answer to "what does this dataset hold". It reports
**coverage** -- which entries exist, not merely whether something does -- because "the
feature ran" and "the feature covers all three sequences" are different facts and only the
second is worth trusting. Every answer is computed from disk at read time, so a view can be
out of date but never wrong.

In [ ]:
inv = inventory(ds)
for rec in inv.records:
    cov = rec.coverage
    print(f"  {rec.name:24s} {rec.status:10s} {len(cov.present)}/{len(cov.target)}")
if not inv.records:
    print("  (nothing computed yet)")

media = pd.read_csv(ds.get_root("media_raw") / "index.csv",
                    keep_default_na=False, dtype=str)
print()
print(media[["group", "sequence", "codec", "width", "height", "fps", "frame_count"]]
      .to_string(index=False))

# Shipped analysis-clean on purpose. Frame extraction routes on this cell: it reads an
# analysis derivative when the verdict says `required`, and `route_media_row` *raises*
# rather than falling back -- so a `required` verdict must be answered, not argued with.
# AV1 is in the frame-exact codec set, so these rows are blank.
still = [v for v in media["analysis_transcode"] if str(v).strip() == "required"]
assert not still, f"{len(still)} clip(s) still require an analysis transcode"
print(f"\nall {len(media)} clips are analysis-clean")

## 2 -- Convert the tracks

`tracks/` is not shipped, because it is entirely regenerable from `tracks_raw/` and
an absent tracks index reads as an empty frame carrying the full schema rather than
raising. So this is the consumer's first real step.

Two details that are easy to get wrong:

- **`multi_sequences_per_file=True`** is already set on the shipped source, and it is
  load-bearing rather than decorative. Only a *blank* `sequence` cell in
  `tracks_raw/index.csv` triggers the converter's `enumerate_sequences`. Without it the
  whole `.npy` collapses into one entry named after the file stem, and nothing fails.
- **`group_from` means two different things** either side of this boundary. On a scan
  source it is `"filename"|"parent"` — where the group comes from. On
  `convert_all_tracks` it is `"infile"|"filename"|"both"` — which one wins.
  `"filename"` is the only value legal on both, and it does not mean the same thing on
  each.

`convert_all_tracks` warns on stderr and keeps going, so a batch that converted
*nothing* looks exactly like one that converted everything. Assert on the outcome.

In [ ]:
ds.scan_tracks()
outcome = ds.convert_all_tracks(group_from="filename")
assert outcome.ok, f"{outcome.failed} file(s) failed to convert"
assert outcome.converted or len(read_tracks_index(ds)), \
    "nothing matched -- check the source patterns and src_format"

tracks = read_tracks_index(ds)

# Pin the variant name now. Section 6 adds a SECOND tracks variant (the TREx run) for
# these same entries, and `select_variant_rows` refuses to guess between two recipes.
# It raises over the WHOLE index, so one ambiguous entry breaks `load_tracks` for
# every sequence -- reading the name here keeps the cells below working afterwards.
CONVERT_VARIANT = sorted(
    tracks.loc[tracks["producer"].astype(str).str.startswith("convert-"), "run_id"].unique()
)[0]

# Entries come from that one variant's rows, not from the whole index. The index
# holds one row per (run_id, group, sequence), so once TREx has run, iterating all
# of it yields the tracked sequence twice -- and section 4 would then emit its
# frames twice. Re-running this notebook top to bottom is what surfaces that.
ENTRIES = [
    (str(r["group"]), str(r["sequence"]))
    for _, r in tracks[tracks["run_id"] == CONVERT_VARIANT].iterrows()
]

print(f"tracks variant: {CONVERT_VARIANT}\n")
print(tracks[["group", "sequence", "run_id", "producer", "n_rows", "n_keypoints"]]
      .to_string(index=False))

`n_keypoints` is measured from the parquet at write time rather than passed in, so no
call site can record a false zero. A blank cell means *unknown*, not zero — keypoints
are optional in `mosaic_v1`, and a centroid-only tracker emits none rather than
copying `X`/`Y` into a fabricated `poseX0`.

What the table does **not** carry is just as deliberate. `mosaic_v1` *forbids* `VX`,
`VY`, `SPEED`, `ANGLE` and the rest: a tracker reports, a feature derives. Heading is
the sharpest case — the principal-component fit a converter would use has an arbitrary
sign, and its flips read downstream as real turns. Anything wanting a heading runs the
`heading` feature and chooses the method, which then enters the run identifier.

In [ ]:
group0, seq0 = ENTRIES[0]
t0 = ds.load_tracks(group0, seq0, run_id=CONVERT_VARIANT)
print(f"{seq0}: {len(t0):,} rows, ids {sorted(t0['id'].unique())}")
print("columns:", ", ".join(t0.columns))
display(t0.head(3))

## 3 -- The frames

The package ships 80 frames per clip, k-means sampled: rather than an even time grid,
k-means over downsampled candidates picks a *spread of distinct-looking* frames, which
matters when the animals are still for long stretches.

Asking for them again is the interesting part. A run is addressed by a hash of its
parameters, so the same request resolves to the same `run_id` — `kmeans-4b56cb1f82` —
finds the directory already there, and returns in milliseconds without decoding a
frame. That is the caching model in one call: nothing is asked "is this up to date",
the name simply already exists.

In [ ]:
frame_run_id = extract_frames(
    ds, n_frames=N_FRAMES, method=FRAME_METHOD,
    candidate_step=5,
    kmeans_resize=(64, 64), kmeans_grayscale=True,
    kmeans_max_candidates=5000, kmeans_batch_size=1024,
    kmeans_max_iter=100, random_state=42,
    parallel_workers="auto", parallel_mode="thread",
)
runs = list_frame_runs(ds, FRAME_METHOD)
this = runs[runs["run_id"] == frame_run_id]
print(f"\nrun_id {frame_run_id}: {len(this)} sequences, "
      f"{this['n_frames_extracted'].astype(int).sum()} frames")

Re-sampling is refused outright rather than overwritten — `overwrite=True` raises
`AnnotatedFramesWouldBeDestroyed`, because frames may already carry hand annotations.
To sample again, mint a new run with `revision`, which the `extract_frames()` wrapper
does not expose:

```python
run_op(ds, "extract-frames",
       {"n_frames": N_FRAMES, "method": "kmeans", "revision": 1}, track=False)
```

### Does frame *N* of the video hold frame *N* of the tracks?

Section 4 joins the extracted frames to the tracks table on the frame index, and that
is the single assumption the whole pose-annotation step rests on. Two checks — one
cheap, one conclusive.

In [ ]:
# Cheap: the video's frame count against the span of the tracks table. If the video
# were trimmed or padded relative to the arrays, every label after the edit would sit
# on the wrong image.
for group, seq in ENTRIES:
    n_video = int(media.loc[media["sequence"] == seq, "frame_count"].iloc[0])
    track = ds.load_tracks(group, seq, run_id=CONVERT_VARIANT)
    n_track = int(track["frame"].max()) + 1
    assert n_video == n_track, (
        f"{seq}: video has {n_video} frames, tracks span {n_track}. "
        "The frame index means different things on the two sides."
    )
    print(f"  {seq.split('__')[-1]}: {n_video} frames on both sides")

In [ ]:
# Conclusive: decode an extracted frame, draw the keypoints joined to it, and look.
import matplotlib.pyplot as plt

N_CHECK = 3
group, seq = ENTRIES[0]
man = get_frame_manifests(ds, FRAME_METHOD, run_id=frame_run_id,
                          group=group, sequence=seq)[0]
track = ds.load_tracks(group, seq, run_id=CONVERT_VARIANT)
picks = man["files"][:: max(1, len(man["files"]) // N_CHECK)][:N_CHECK]

fig, axes = plt.subplots(1, len(picks), figsize=(5 * len(picks), 5))
for ax, rec in zip(np.atleast_1d(axes), picks):
    fidx = int(rec["frame_index"])
    ax.imshow(plt.imread(rec["path"]))
    for _, r in track[track["frame"] == fidx].iterrows():
        xs = [r[f"poseX{k}"] for k in range(len(KEYPOINT_NAMES))]
        ys = [r[f"poseY{k}"] for k in range(len(KEYPOINT_NAMES))]
        for a, b in SKELETON:
            ax.plot([xs[a], xs[b]], [ys[a], ys[b]], "-", lw=1.2, color="#ff00ff")
        ax.scatter(xs, ys, s=18, c="#00ff88", zorder=3, edgecolors="black", linewidths=.4)
    ax.set_title(f"frame {fidx}")
    ax.axis("off")
fig.suptitle(f"{seq.split('__')[-1]} -- keypoints joined by frame index", y=1.02)
plt.tight_layout()
plt.show()
print("If the markers sit on the mice, frame index and pose index agree.")

## 4 -- Tracks -> pose pseudo-annotations

CalMS21 ships MARS-derived keypoints for every frame. Treating them as ground truth
for a *new* pose model is bootstrapping, not annotation — the labels are as good as
MARS was, no better. That is fine for an example, and it is the honest framing: it
demonstrates the training path end to end without anyone hand-labelling a frame.

The join is on the frame index. `extract_frames` records it three ways — in the PNG
name, in `run_info.json`'s `files[i]["frame_index"]`, and in `selected_frame_indices` —
and it is the 0-based contiguous video frame index, which is exactly the tracks table's
`frame` column.

Everything goes through `AnnotationSet` rather than straight to YOLO text. That is the
canonical in-memory shape for "labelled frames"; the YOLO writer is one emitter off it,
CVAT and COCO are others, and keeping the intermediate means a second target costs a
writer rather than a rewrite.

In [ ]:
SCHEMA = KeypointSchema(names=KEYPOINT_NAMES, skeleton=SKELETON)
POSE_X = [f"poseX{k}" for k in range(len(KEYPOINT_NAMES))]
POSE_Y = [f"poseY{k}" for k in range(len(KEYPOINT_NAMES))]

frames_out, video_of_image = [], {}
for group, seq in ENTRIES:
    mans = get_frame_manifests(ds, FRAME_METHOD, run_id=frame_run_id,
                               group=group, sequence=seq)
    assert len(mans) == 1, f"{group}/{seq}: expected 1 manifest, got {len(mans)}"
    man = mans[0]
    stem = seq.split("__")[-1]

    track = ds.load_tracks(group, seq, run_id=CONVERT_VARIANT)
    by_frame = {int(f): sub for f, sub in track.groupby("frame")}

    for rec in man["files"]:
        fidx = int(rec["frame_index"])
        rows = by_frame.get(fidx)
        if rows is None or rows.empty:
            continue
        objects = []
        for _, r in rows.iterrows():
            kps = tuple(
                Keypoint(x=float(r[xc]), y=float(r[yc]), visibility=2)
                for xc, yc in zip(POSE_X, POSE_Y)
                if np.isfinite(r[xc]) and np.isfinite(r[yc])
            )
            if len(kps) != len(KEYPOINT_NAMES):
                continue  # partial pose: drop the instance, keep the frame
            objects.append(AnnotationObject(keypoints=kps, category="mouse",
                                            track_id=str(int(r["id"]))))
        if not objects:
            continue
        # Renamed per video so the flat images/ directory stays unique and each
        # frame's source clip is still readable from its filename.
        name = f"{stem}__frame_{fidx:06d}.png"
        video_of_image[name] = stem
        frames_out.append(AnnotationFrame(
            image_path=Path(rec["path"]).with_name(name),
            width=int(rec["width"]), height=int(rec["height"]),
            objects=tuple(objects), video=stem, frame_index=fidx,
        ))

aset = AnnotationSet(schema=SCHEMA, frames=tuple(frames_out), categories=("mouse",))
print(f"{len(aset.frames)} annotated frames, "
      f"{sum(len(f.objects) for f in aset.frames)} instances")

### Emit the YOLO pose tree

Three choices worth naming.

**`BboxPolicy(method="isotropic")`.** A box tight to the keypoint hull collapses to
zero height whenever a mouse is captured nose-to-tail in a straight line, and a
zero-area box is a broken training target. The isotropic policy pads relative to body
length instead.

**A per-video split.** Three clips cannot fill train/valid/test group-wise without
leaving one empty, which Ultralytics treats as a hard error. So it is 2 videos train,
1 valid, no test — group-aware, so no frame of the validation animal is ever seen in
training, and validation mAP is a real across-animal estimate rather than a leaked one.

**`flip_idx` written by hand.** `make_data_yaml` omits it, and without it Ultralytics
silently sets `fliplr=flipud=0` — the horizontal-flip augmentation you asked for simply
does not happen, with no warning, because flipping an image without swapping
`left_ear`/`right_ear` would teach the model a mirrored anatomy.

In [ ]:
VIDEO_STEMS = [seq.split("__")[-1] for _, seq in ENTRIES]
TRAIN_VIDS = set(VIDEO_STEMS[:N_TRAIN_VIDS])
split_of = {name: ("train" if vid in TRAIN_VIDS else "valid")
            for name, vid in video_of_image.items()}

# Copy each extracted PNG to its annotation name, so `usable_frames` finds it.
for f in aset.frames:
    if not f.image_path.exists():
        shutil.copy2(f.image_path.with_name(f"frame_{f.frame_index:06d}.png"), f.image_path)

YOLO_DIR = DATASET / "pose_dataset"
shutil.rmtree(YOLO_DIR, ignore_errors=True)
policy = BboxPolicy(method="isotropic", pad_frac_of_body=0.30, min_pad_px=20.0)

written, skipped = write_split_tree(
    usable_frames(aset), YOLO_DIR, split_of,
    lambda fr: [ln for ln in
                (yolo_pose_line(o, fr.width, fr.height, class_id=0, policy=policy)
                 for o in fr.objects) if ln],
    symlink_images=False,
)

data_yaml = Path(make_data_yaml(YOLO_DIR, {"mouse": 0},
                                kpt_shape=[len(KEYPOINT_NAMES), 3]))
text = data_yaml.read_text()
if "flip_idx" not in text:
    data_yaml.write_text(text.rstrip() + f"\nflip_idx: {FLIP_IDX}\n")

for sub in ("train", "valid"):
    n = len(list((YOLO_DIR / sub / "images").glob("*.png")))
    print(f"  {sub}: {n} images  ({', '.join(sorted(TRAIN_VIDS if sub == 'train' else set(VIDEO_STEMS) - TRAIN_VIDS))})")
    assert n > 0, f"{sub} split is empty -- Ultralytics will fail to load it"
print(f"written={written} skipped={skipped}\n")
print(data_yaml.read_text())

### Check the labels before training

Ultralytics drops a label file whose keypoints fall outside `[0, 1.01]` and reports it
only as `corrupt` in a log line, so a silently half-empty training set is an easy
mistake to make. `yolo_pose_line` clips, but verifying costs nothing.

In [ ]:
NK, NDIM = len(KEYPOINT_NAMES), 3
bad = []
for lbl in sorted(YOLO_DIR.rglob("labels/*.txt")):
    for i, line in enumerate(lbl.read_text().splitlines()):
        v = np.array(line.split(), dtype=float)
        # Ultralytics asserts this width: class + bbox + one (x, y, visibility)
        # triplet per keypoint.
        if v.size != 5 + NK * NDIM:
            bad.append((lbl.name, i, f"width {v.size}, expected {5 + NK * NDIM}"))
            continue
        kp = v[5:].reshape(NK, NDIM)
        coords = np.concatenate([v[1:5], kp[:, :2].ravel()])
        if coords.min() < -0.01 or coords.max() > 1.01:
            bad.append((lbl.name, i, float(coords.min()), float(coords.max())))
        # The third value of each triplet is visibility (0/1/2), not a coordinate.
        if not set(np.unique(kp[:, 2]).tolist()) <= {0.0, 1.0, 2.0}:
            bad.append((lbl.name, i, "visibility", np.unique(kp[:, 2]).tolist()))
        if (v[3] <= 0) or (v[4] <= 0):
            bad.append((lbl.name, i, "degenerate box", float(v[3]), float(v[4])))

n_img = len(list(YOLO_DIR.rglob("images/*.png")))
n_lbl = len(list(YOLO_DIR.rglob("labels/*.txt")))
assert not bad, bad[:5]
assert n_img == n_lbl, f"{n_img} images vs {n_lbl} labels"
print(f"{n_img} images each with a label -- all keypoints in range")

## 5 -- Train the pose model

Through the `train-pose` op rather than Ultralytics directly, so the run gets a
`run_id`, a row in `models/train-pose/index.csv`, and a reuse gate. The weights land
at `<run>/train/weights/best.pt`.

The op fingerprints the **contents** of the YOLO tree, so changing the frame set or
the split mints a new `run_id` and retrains from scratch. That is correct, and it is
why re-running this cell after re-emitting section 4 is not free.

`PRETRAINED_MODEL` in section 0 skips all of this and hands section 6 a checkpoint you
already have. `TRAIN = False` with no `PRETRAINED_MODEL` skips it too, and section 6 then
tracks with no detector -- which works, and produces no keypoints.

### First, make sure the base checkpoint is intact

Ultralytics decides a checkpoint is already downloaded by asking whether a file of
that name **exists** — no size check, no checksum, no trial load. So an interrupted
download (a restarted kernel, a dropped connection) leaves a partial file that is
served as a cache hit *forever*, and the only symptom is `PytorchStreamReader failed
reading zip archive` from deep inside `torch.load`.

`safe_download` does unlink a partial on terminal failure, precisely so this cannot
happen — but that cleanup needs the process to survive long enough to run it. A killed
kernel skips it. So: actually load the file, and re-fetch if it will not.

In [ ]:
BASE_MODEL_PATH = None
if TRAIN and PRETRAINED_MODEL is None:
    import torch
    from ultralytics.utils.downloads import attempt_download_asset

    def ensure_checkpoint(name: str) -> str:
        """Return a path to *name* that torch can actually load."""
        for attempt in ("cached", "refetched"):
            path = Path(attempt_download_asset(name))
            try:
                torch.load(path, map_location="cpu", weights_only=False)
                print(f"{name}: {path} ({path.stat().st_size / 1e6:.1f} MB, {attempt})")
                return str(path)
            except Exception as exc:
                if attempt == "refetched":
                    raise
                print(f"{name} at {path} will not load ({type(exc).__name__}); re-fetching")
                path.unlink(missing_ok=True)
        raise AssertionError("unreachable")

    BASE_MODEL_PATH = ensure_checkpoint(BASE_MODEL)
else:
    print("not training here -- skipping the checkpoint preflight")

In [ ]:
TRAIN_RUN_ID = None
MODEL_WEIGHTS = None
METRICS_CSV = None

if TRAIN and PRETRAINED_MODEL is None:
    TRAIN_RUN_ID = run_op(ds, "train-pose", {
        "data": str(data_yaml),
        "model": BASE_MODEL_PATH,
        "epochs": EPOCHS,
        "imgsz": IMGSZ,
        "patience": 50,
        "augmentation": AUGMENT,
        "device": DEVICE,   # HASH_EXCLUDE -- retuning it does not bust the cache
        "batch": BATCH,     # HASH_EXCLUDE
    })
    models = pd.read_csv(ds.get_root("models") / "train-pose" / "index.csv",
                         keep_default_na=False, dtype=str)
    row = models[models["run_id"] == TRAIN_RUN_ID].iloc[0]
    MODEL_WEIGHTS = ds.resolve_path(row["best_model_path"])
    METRICS_CSV = ds.resolve_path(row["metrics_path"])
    print(f"train run: {TRAIN_RUN_ID}  status={row['status']} epochs={row['n_epochs']}")
    print("weights:", MODEL_WEIGHTS, MODEL_WEIGHTS.exists())
elif PRETRAINED_MODEL is not None:
    MODEL_WEIGHTS = Path(PRETRAINED_MODEL)
    assert MODEL_WEIGHTS.exists(), MODEL_WEIGHTS
    # <run>/train/weights/best.pt -> <run>/train/results.csv, if it came from an
    # Ultralytics run at all. Absent is fine; only the curves below need it.
    sidecar = MODEL_WEIGHTS.parents[1] / "results.csv"
    METRICS_CSV = sidecar if sidecar.exists() else None
    print("using the supplied checkpoint:", MODEL_WEIGHTS)
else:
    print("skipped: no model, so section 6 tracks without a detector")

### Training curves

The pair to watch is **pose** — the keypoint regression, and the thing this model
exists to do. With two training animals and one validation animal, train and val
separating is the expected signature of a model memorising two mice rather than
learning "mouse". A gap that opens and keeps widening says to add clips before adding
epochs.

In [ ]:
df = None
if METRICS_CSV is not None and Path(METRICS_CSV).exists():
    df = pd.read_csv(METRICS_CSV)
    df.columns = [c.strip() for c in df.columns]
    cols = [c for c in df.columns if "mAP" in c or "pose" in c.lower()][:4]
    display(df[["epoch"] + cols].tail(10))
else:
    print("no results.csv to read -- nothing was trained here")

In [ ]:
def plot_curves(df, title):
    """Train vs validation loss for every loss Ultralytics logged, plus the mAPs."""
    import matplotlib.pyplot as plt

    LOSSES = [("box_loss", "box"), ("pose_loss", "pose"), ("kobj_loss", "keypoint obj"),
              ("cls_loss", "class"), ("dfl_loss", "dfl")]
    present = [(k, lab) for k, lab in LOSSES
               if f"train/{k}" in df.columns and f"val/{k}" in df.columns]

    fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True)
    flat = axes.ravel()
    for ax, (key, label) in zip(flat, present):
        ax.plot(df["epoch"], df[f"train/{key}"], label="train", lw=1.6)
        ax.plot(df["epoch"], df[f"val/{key}"], label="validation", lw=1.6, ls="--")
        ax.set_title(label)
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.3)
        if ax is flat[0]:
            ax.legend(frameon=False)

    ax = flat[len(present)]
    for col, lab in [("metrics/mAP50(P)", "pose mAP50"),
                     ("metrics/mAP50-95(P)", "pose mAP50-95"),
                     ("metrics/mAP50(B)", "box mAP50")]:
        if col in df.columns:
            ax.plot(df["epoch"], df[col], label=lab, lw=1.6)
    ax.set_title("validation mAP")
    ax.set_xlabel("epoch")
    ax.legend(frameon=False, fontsize=8)
    ax.grid(alpha=0.3)
    for ax in flat[len(present) + 1:]:
        ax.axis("off")

    fig.suptitle(title, y=1.0)
    plt.tight_layout()
    plt.show()

if df is not None:
    plot_curves(df, f"YOLO pose training -- {TRAIN_RUN_ID or MODEL_WEIGHTS}")

## 6 -- Track with TREx, using that model

This is what closes the loop: the pose model becomes TREx's *detector*, and the footage
gets tracked by a model this dataset produced. `TREX_ENTRIES` in section 0 decides how
many clips — one, by default, because the cost below is per clip.

**Three settings carry the interesting behaviour.**

`detect_model` hands TREx the YOLO pose model. Without it TREx falls back to
background subtraction, and the resulting table carries **no `poseX*`/`poseY*` columns
at all** — the overlay in sections 7-8 would then draw a bare centroid ring per animal,
with no skeleton. For a notebook about a pose model that is a thin payoff.

`auto_train=True` turns on TREx's **visual identification**: it learns each
individual's appearance and uses that to keep identities across crossings, rather than
relying on motion continuity alone. CalMS21 is close to the ideal case, because the
resident and intruder differ in coat colour — a black mouse and a white one are about
as separable as visual identity gets. On animals that look alike it helps much less and
costs a training pass.

`track_max_individuals=2` is worth pinning once and leaving alone: it is a term in
*both* the conversion and the tracking digests, so changing it later forces a full
re-detection pass rather than reusing the conversion.

In [ ]:
# Whatever section 5 left behind. `detect_model` takes a bare weights path as well
# as a training run_id: a run identifier carries lineage and is preferred, while a
# path is identified by the content digest of its bytes -- which is why swapping
# the file in place still mints a different TREx run rather than reusing this one.
DETECT_MODEL = str(PRETRAINED_MODEL) if PRETRAINED_MODEL else TRAIN_RUN_ID
print("detect_model:", DETECT_MODEL or "(none -- background subtraction)")

### Locate TREx without running it

Probe the environment **before** spending a conversion pass: the not-found error is
raised lazily inside the first entry's convert phase, after a run root, a tracks
variant and a failed run-log have already been written.

`_trex_invocation()` honours the whole location ladder and runs nothing, which is the
only safe way to ask. Do not "check" by calling `trex -h` or `trex -version`: on macOS
both open a window and block.

In [ ]:
import os

if TREX_CONDA_ENV:
    os.environ["MOSAIC_TREX_CONDA_ENV"] = TREX_CONDA_ENV
# Alternatives, if TREx is not a conda environment:
# os.environ["MOSAIC_TREX_BIN"] = "/abs/path/to/trex"
# os.environ["MOSAIC_TREX_DISPLAY"] = ":99"   # headless Linux: Xvfb

from mosaic.tracking.trex.run import TRexNotFoundError, _trex_invocation

try:
    TREX_ARGV = _trex_invocation()   # resolves; runs nothing
    print("TREx:", " ".join(str(a) for a in TREX_ARGV))
except TRexNotFoundError as exc:
    TREX_ARGV = None
    print(f"TREx unavailable -- section 6 is skipped, and sections 8-9 fall back\n"
          f"to the CalMS21 tracks.\n  {exc}")

### The model has to be loadable *by TREx*, not just by us

TREx runs in its own environment with its own `ultralytics`, and it loads the
checkpoint by unpickling it — which resolves the model's head class **by name**. So the
two installs have to agree about that name.

This is a real constraint on `BASE_MODEL`, not a theoretical one. A `yolo26*-pose`
checkpoint carries a `Pose26` head; an older ultralytics has only `Pose`, and loading
aborts with `Can't get attribute 'Pose26'` — as a SIGABRT out of a C++ exception, after
the video has already opened, which reads like a video problem and is not one.

Checking costs one subprocess. Failing costs a full conversion pass.

In [ ]:
TREX_MODEL_OK = False
if TREX_ARGV is not None and DETECT_MODEL:
    # The environment's python sits beside its trex binary. This runs *python*,
    # not trex, so it opens no window.
    trex_python = Path(str(TREX_ARGV[-1])).parent / "python"
    probe_src = (
        "import sys, ultralytics;"
        "from ultralytics import YOLO;"
        "m = YOLO(sys.argv[1]);"
        "print('ultralytics', ultralytics.__version__,"
        "      '| head', type(m.model.model[-1]).__name__)"
    )
    probe = subprocess.run([str(trex_python), "-c", probe_src, str(MODEL_WEIGHTS)],
                           capture_output=True, text=True)
    import ultralytics as _ul
    print(f"  ours : ultralytics {_ul.__version__}")
    print(f"  TREx : {probe.stdout.strip() or '(failed)'}")
    if probe.returncode == 0:
        TREX_MODEL_OK = True
    else:
        tail = probe.stderr.strip().splitlines()[-1] if probe.stderr.strip() else ""
        print(f"\nTREx cannot load this checkpoint:\n  {tail}\n"
              "  Section 6 will track without it (background subtraction), which\n"
              "  produces no keypoints. To use the model, train with a BASE_MODEL\n"
              "  whose head class TREx's ultralytics knows -- or upgrade it there.")

TREx is handed a video *path*, and these clips are AV1. Check its environment can
decode AV1 before committing to a conversion pass:

```bash
conda run -n track ffmpeg -hide_banner -decoders | grep av1
```

The run below writes raw TREx output under `_tracking/trex/<run_id>/`, then bridges it
into `tracks/<variant>/`. `_tracking/` is a working root: `mosaic sweep-tracking`
reclaims it once a run is finished and past its retention window, and it is excluded by
name from every scan that walks the dataset for user content.

**Watch the disk.** The `.pv` TREx converts to is close to raw — about 320 KB a frame
at this resolution, so the 1800-frame window below costs ~580 MB and the whole
21 364-frame clip would cost **~6.8 GB**. That, not detection time, is the main reason
`TREX_CONVERSION_RANGE` exists. The conversion is shared across tracking-parameter
sweeps, so it is paid once per detection recipe rather than once per run.

In [ ]:
TREX_VARIANT = None
if TREX_ARGV is not None:
    scope = ENTRIES if TREX_ENTRIES is None else ENTRIES[:TREX_ENTRIES]
    TREX_PARAMS = {
        "entries": [f"{g}:{s}" for g, s in scope],   # "group:sequence" tokens
        # Only hand TREx the model if TREx can actually load it; otherwise fall back
        # to background subtraction rather than aborting mid-conversion.
        **({"detect_model": str(DETECT_MODEL), "detect_type": "yolo"}
           if TREX_MODEL_OK else {}),
        "track_max_individuals": 2,
        "auto_train": True,          # visual identification
        "idle_timeout": 7200,        # HASH_EXCLUDE: not part of the run_id
        # Without this the table publishes with no keypoints and nothing says so.
        "detect_keypoint_count": len(KEYPOINT_NAMES),
    }
    if TREX_CONVERSION_RANGE is not None:
        # A pass-through into TREx's own settings, and a *convert* key -- so it is
        # part of the conversion digest, and widening the range later reconverts
        # rather than silently reusing a short `.pv`.
        TREX_PARAMS["convert_extra_settings"] = {
            "video_conversion_range": list(TREX_CONVERSION_RANGE)
        }

    print("tracking:", ", ".join(s for _, s in scope))
    TREX_VARIANT = run_op(ds, "trex", TREX_PARAMS)
    print("TREx run / tracks variant:", TREX_VARIANT)
else:
    print("skipped")

### Reading a table when two variants exist

An entry now carries two recipes, and mosaic refuses to guess between them —
`select_variant_rows` raises rather than picking. Two details, because both bite:

- The refusal is computed over the **whole index**, so one ambiguous entry breaks
  `load_tracks` for every sequence, not just that one.
- The keyword differs by call site. `load_tracks` and `drop_entries` take **`run_id=`**;
  `run_feature` and `build_manifest` take **`tracks_run_id=`**.

And pass a name read from the index, never one typed by hand: an unknown variant does
not raise, it falls through to the auto-convert arm and quietly re-converts.

In [ ]:
idx = read_tracks_index(ds)
print(idx[["group", "sequence", "run_id", "producer", "n_rows", "n_keypoints"]]
      .to_string(index=False))

if TREX_VARIANT:
    group, seq = ENTRIES[0]
    from_calms = ds.load_tracks(group, seq, run_id=CONVERT_VARIANT)
    from_trex = ds.load_tracks(group, seq, run_id=TREX_VARIANT)
    print()
    for name, tbl in (("CalMS21 arrays", from_calms), ("TREx + our model", from_trex)):
        kp = len([c for c in tbl.columns if c.startswith("poseX")])
        print(f"  {name:18s} {len(tbl):8,} rows  {tbl['id'].nunique()} ids  "
              f"{kp} keypoints  frames {int(tbl['frame'].min())}-{int(tbl['frame'].max())}")
    if TREX_MODEL_OK:
        assert len([c for c in from_trex.columns if c.startswith("poseX")]), (
            "TREx tracked with a pose model but exported no keypoints -- check "
            "detect_keypoint_count (see the cell above)."
        )

## 7 -- Tracked frames from across the recording

Sampled evenly across the tracked span rather than from one stretch, because the
failure a tracker actually has is an identity swap after a crossing, and that is
invisible in five consecutive frames.

These are rendered by the same code that writes the video in section 8, so what you see
here is exactly what gets encoded.

In [ ]:
import matplotlib.pyplot as plt

from mosaic.behavior.visualization_library.playback import build_overlay
from mosaic.behavior.visualization_library.video_stream import render_stream

N_STILLS = 5
SHOW_VARIANT = TREX_VARIANT or CONVERT_VARIANT
group, seq = ENTRIES[0]

overlay_data, tracks_df, _ = build_overlay(
    ds, group=group, sequence=seq,
    feature_runs={},
    label_kind=None,          # no converted behaviour labels in this dataset
    tracks_run_id=SHOW_VARIANT,
)

resolved = ds.resolve_media(group, seq)
# The tracked span, not the whole video: with TREX_CONVERSION_RANGE set, the table
# stops well before the recording does and the later stills would be empty.
lo, hi = int(tracks_df["frame"].min()), int(tracks_df["frame"].max())
picks = np.linspace(lo, hi, N_STILLS, dtype=int).tolist()

fig, axes = plt.subplots(1, N_STILLS, figsize=(4.2 * N_STILLS, 4.4))
for ax, k in zip(np.atleast_1d(axes), picks):
    # start == end yields exactly one frame; the stream seeks rather than scanning.
    stream = render_stream(resolved.paths, overlay_data,
                           start=int(k), end=int(k), facts=resolved.facts)
    frame_idx, frame = next(iter(stream))
    stream.close()
    ax.imshow(frame[:, :, ::-1])          # BGR -> RGB
    ax.set_title(f"frame {frame_idx}")
    ax.axis("off")
fig.suptitle(f"{seq} -- {SHOW_VARIANT}", y=1.03)
plt.tight_layout()
plt.show()

## 8 -- An annotated video

`start` and `end` are **params of the feature**, not filters on the run, and that
distinction matters: `run_feature`'s frame filters narrow only the DataFrame handed to
`apply()`, while the renderer re-reads the full table off disk and encodes whatever
`params.start`/`params.end` say. Filtering the run would move the `run_id` *and* still
encode every frame.

`end` is inclusive, and both are absolute frame indices, so a window taken from the
middle of a recording draws the right annotations.

The overlay is a **feature**, categorised `media` — so the rendered video is addressed
by `run_id` like every other artifact, and a pipeline can end on the thing a biologist
actually looks at rather than one step short of it.

In [ ]:
from mosaic.behavior.visualization_library import Overlay
from mosaic.core.pipeline.index import feature_run_root

FPS = float(media.loc[media["sequence"] == seq, "fps"].iloc[0])
CLIP_START = lo
CLIP_END = min(hi, CLIP_START + int(round(FPS * CLIP_SECONDS)) - 1)   # inclusive

overlay_result = ds.run_feature(
    Overlay(params={
        "label_kind": None,   # the default is "behavior"; without labels that only
                              # prints a warning per entry
        "start": CLIP_START,
        "end": CLIP_END,
        "downscale": 1.0,
        "draw_options": {"point_radius": 4, "bbox_thickness": 2},
    }),
    entries=[(group, seq)],
    tracks_run_id=SHOW_VARIANT,
)
print(overlay_result)

overlay_mp4 = (feature_run_root(ds, overlay_result.feature, overlay_result.run_id)
               / f"{make_entry_key(group, seq)}.mp4")
print(overlay_mp4, overlay_mp4.exists(), f"{overlay_mp4.stat().st_size / 1e6:.1f} MB")

The overlay is written with OpenCV's `mp4v`, which most browsers will not play. A quick
H.264 re-encode makes it viewable inline and shareable.

In [ ]:
overlay_h264 = overlay_mp4.with_name("overlay_h264.mp4")
subprocess.run(
    ["ffmpeg", "-y", "-loglevel", "error", "-i", str(overlay_mp4),
     "-c:v", "libx264", "-crf", "20", "-pix_fmt", "yuv420p", str(overlay_h264)],
    check=True,
)
size_mb = overlay_h264.stat().st_size / 1e6
print(overlay_h264, f"{size_mb:.1f} MB")

# Embedding base64-encodes the clip into the .ipynb, so only do it while it is small --
# a committed notebook carrying 15 MB of video is its own problem.
EMBED_LIMIT_MB = 8
try:
    from IPython.display import Video
    if size_mb <= EMBED_LIMIT_MB:
        display(Video(str(overlay_h264), embed=True, width=720))
    else:
        print(f"not embedding ({size_mb:.0f} MB > {EMBED_LIMIT_MB} MB); open it from "
              "the path above, or lower CLIP_SECONDS")
except ImportError:
    print("(run under Jupyter to see it inline)")

## 9 -- Where to go from here

What this notebook covers, in order:

1. CalMS21's arrays become standard tracks, and those tracks become **pseudo-annotations**
   -- no manual labelling at all.
2. A YOLO pose model trains on 240 frames from three recordings, validated on a **held-out
   animal**, so the score answers "does this transfer to a mouse it never saw".
3. TREx tracks with that model as its detector, with visual identification on.
4. The result is rendered back over the footage.

Everything derived is addressed by a hash of what produced it, so re-running any cell above
is a cache hit until a parameter actually changes, and a parameter sweep organises itself.

**The tracks this produces are not good tracks, and that is expected.** With the default
parameters used here, TREx tends to drop a tracklet when the two mice come close -- which is
exactly the moment a resident-intruder assay is about -- so the output is visibly worse than
the CalMS21 tables it was trained on, and those had been corrected. Getting tracks worth
analysing needs parameter adjustment and probably a better pose model than 240 bootstrapped
frames can support. That is a separate exercise, deliberately not attempted here, and worth
revisiting as its own step.

**The point is the shape of the workflow**, not the quality of this particular result: that
tracks can seed annotations, that annotations train a model, that the model drives a
tracker, and that every artifact along the way is addressed and reproducible. Read the
numbers as a demonstration that the path runs end to end, not as a result to build on.

Natural next steps, roughly in order of payoff:

- **Raise `track_max_speed` and lower `track_trusted_probability`**, and watch where the
  identity switches move. Both are tracking settings, so each variant reuses the `.pv` and
  costs one tracking pass rather than another detection pass.
- **Train on more frames.** 80 per clip was chosen to keep the download small, not because
  it is enough. `revision=1` on `extract-frames` mints a fresh sample without destroying the
  one that shipped.
- **Score the two variants against each other.** The CalMS21 tables are ground truth of a
  sort, and where they and TREx disagree is where to look -- crossings, mostly.
- **Add features.** `heading` (`method="two_point"`, `front_idx=3`, `rear_idx=6`) derives the
  body angle these tracks deliberately do not carry; `speed-angvel` and `nearest-neighbor`
  build on it.
- **Run it as one graph.** Sections 2-8 are a pipeline; written as a recipe,
  `mosaic pipeline plan --recipe @file.json` says what each step would be called and what is
  already done, before anything runs.

**Reclaiming the disk.** `_tracking/` holds TREx's `.pv` conversions, which are the largest
thing here by far -- `mosaic sweep-tracking -m <dataset>/dataset.yaml` reclaims the finished
ones. `RESET_DERIVED = True` in section 0 drops every derived root and starts over.